In [0]:
from datetime import datetime, timezone
import uuid
from pyspark.sql import functions as F

QUALITY_TABLE = "health_and_habit_guardian.silver.data_quality"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {QUALITY_TABLE} (
    run_id          STRING,
    table_name      STRING,
    run_timestamp   TIMESTAMP,
    total_rows      LONG,
    valid_rows      LONG,
    invalid_rows    LONG,
    null_user_ids   LONG,
    duplicate_rows  LONG,
    min_date        DATE,
    max_date        DATE,
    quality_status  STRING
) USING DELTA
""")


In [0]:
def _run_id() -> str:
    """Retrieves the Databricks Run ID on Serverless compute with type safety."""
    try:
        # Internal context manager supported natively on Databricks Serverless
        from dbruntime.databricks_repl_context import get_context
        ctx = get_context()
        if ctx and hasattr(ctx, "idInJob") and ctx.idInJob:
            return str(ctx.idInJob)
    except (ImportError, AttributeError, Exception):
        # Fallback if executed outside of Databricks context (e.g., local unit tests)
        pass

    return str(uuid.uuid4())


def write_quality_summary(
    df,
    table_name,
    date_col="date",
    dedup_cols=None,
    invalid_threshold=0.05,
):
    m = (
        df.agg(
            F.count("*").alias("total_rows"),
            F.sum(
                F.when(F.col("_quality_status") == "VALID", 1).otherwise(0)
            ).alias("valid_rows"),
            F.sum(
                F.when(F.col("_quality_status") == "INVALID", 1).otherwise(0)
            ).alias("invalid_rows"),
            F.sum(
                F.when(F.col("user_id").isNull(), 1).otherwise(0)
            ).alias("null_user_ids"),
            F.min(F.col(date_col).cast("date")).alias("min_date"),
            F.max(F.col(date_col).cast("date")).alias("max_date"),
        )
    ).collect()[0]

    duplicates = 0
    if dedup_cols and m["total_rows"] > 0:
        duplicates = df.count() - df.select(*dedup_cols).dropDuplicates().count()

    if m["total_rows"] == 0:
        status = "EMPTY"
    elif (
        m["null_user_ids"] > 0 or
        m["invalid_rows"] / m["total_rows"] > invalid_threshold
    ):
        status = "FAIL"
    elif m["invalid_rows"] > 0:
        status = "WARNING"
    else:
        status = "PASS"

    row = (
        _run_id(),
        table_name,
        datetime.now(timezone.utc),
        m["total_rows"],
        m["valid_rows"],
        m["invalid_rows"],
        m["null_user_ids"],
        duplicates,
        m["min_date"],
        m["max_date"],
        status,
    )

    spark.createDataFrame(
        [row],
        [
            "run_id",
            "table_name",
            "run_timestamp",
            "total_rows",
            "valid_rows",
            "invalid_rows",
            "null_user_ids",
            "duplicate_rows",
            "min_date",
            "max_date",
            "quality_status",
        ],
    ).write.format("delta").mode("append").saveAsTable(QUALITY_TABLE)
